# ERE - Eastern Roman Emperors Rating Project

Edit scores in ERE_List_20260226.csv, then run Section 5. Leave Difficulty and Effectiveness blank to place an emperor in the Unrankable box.

## 0 - Imports

In [7]:
import pandas as pd
import requests
import json
import base64
import os
import time
from urllib.parse import urlparse


## 1 - Load Data

ERE_List_20260226.csv is the master file.

In [8]:
df = pd.read_csv('ERE_List_20260226.csv')
print(f'Loaded {len(df)} emperors')
df.head()

Loaded 90 emperors


,List_Number,Simple_Name,Simple_Name_With_Epithets,Full_Name,Broad_Dynasty,Family_Name,Reign_Start,Reign_End,Reign_Length_Years,Summary,...,Image_URL,Local_Image,Wiki_Length,Wiki_Views,Length_Rank,Views_Rank,Wiki_Score,Difficulty,Effectiveness,Composite
0,1,Constantine I,Constantine I the Great,Flavius Valerius Constantinus,Constantinian,Constantinus,306,337,31,Sole emperor from 324. Moved the imperial capi...,...,https://upload.wikimedia.org/wikipedia/commons...,images/001_.jpg,180856,4576,1.0,1.0,1.0,11.0,20.0,82.0
1,2,Constantius II,Constantius II,Flavius Julius Constantius,Constantinian,Constantinus,337,361,24,Son of Constantine I. Sole emperor from 350. F...,...,https://upload.wikimedia.org/wikipedia/commons...,images/002_.jpg,68764,716,7.0,8.0,7.5,6.0,8.0,36.0
2,3,Julian,Julian the Apostate,Flavius Claudius Julianus,Constantinian,Constantinus,361,363,2,Last non-Christian emperor. Attempted to resto...,...,https://upload.wikimedia.org/wikipedia/commons...,images/003_.jpg,110060,1170,2.0,4.0,3.0,10.0,12.0,56.0
3,4,Jovian,Jovian,Flavius Jovianus,Non-dynastic,Non-dynastic,363,364,1,Elected by the army after Julian's death. Made...,...,https://upload.wikimedia.org/wikipedia/commons...,images/004_.jpg,909,35,79.0,82.0,80.5,12.0,10.0,54.0
4,5,Valens,Valens,Flavius Valens,Valentinianic,Valentinianus,364,378,14,"Eastern emperor, brother of Valentinian I. All...",...,https://upload.wikimedia.org/wikipedia/commons...,images/005_.png,47541,477,17.0,12.0,14.5,11.0,6.0,40.0


## 2 - View Current Ratings

Read-only. Edit scores in the CSV directly.

In [9]:
import pandas as pd

df = pd.read_csv('ERE_List_20260226.csv')
print(f"{'#':<4} {'Emperor':<45} {'D':>4} {'E':>4}")
print('-' * 60)
for _, row in df.iterrows():
    d = int(row['Difficulty']) if str(row['Difficulty']) != 'nan' else '--'
    e = int(row['Effectiveness']) if str(row['Effectiveness']) != 'nan' else '--'
    print(f"{int(row['List_Number']):<4} {row['Simple_Name_With_Epithets']:<45} {str(d):>4} {str(e):>4}")


#    Emperor                                          D    E
------------------------------------------------------------
1    Constantine I the Great                         11   20
2    Constantius II                                   6    8
3    Julian the Apostate                             10   12
4    Jovian                                          12   10
5    Valens                                          11    6
6    Theodosios I the Great                          16   16
7    Arkadios                                         8    5
8    Theodosios II                                    9   13
9    Markianos                                       12   12
10   Leo I the Thracian                               8   11
11   Leo II                                          --   --
12   Zeno                                            15   16
13   Basiliskos                                       8    6
14   Anastasios I                                    10   18
15   Justin I           

## 3 - Fetch Wikipedia Metrics

Takes ~2 min. Only re-run when needed.

In [10]:
import pandas as pd
import requests
import time

df = pd.read_csv('ERE_List_20260226.csv')

session = requests.Session()
session.headers.update({
    'User-Agent': 'ByzantineEmperorsTool/1.0 (educational project; you@example.com)'
})

# Manual Wikipedia title overrides where the name won't match automatically
WIKI_TITLES = {
    'Constantine I the Great':          'Constantine the Great',
    'Justinian I the Great':            'Justinian I',
    'Theodosios I the Great':           'Theodosius I',
    'Theodosios II':                    'Theodosius II',
    'Markianos':                        'Marcian',
    'Anastasios I':                     'Anastasius I Dicorus',
    'Herakleios':                       'Heraclius',
    'Constantine III':                  'Constantine III (emperor)',
    'Heraklonas':                       'Heraclonas',
    'Constans II':                      'Constans II',
    'Constantine IV':                   'Constantine IV',
    'Justinian II Slit-Nose':           'Justinian II',
    'Leontios':                         'Leontius (emperor)',
    'Tiberios II the Constantine':      'Tiberius II Constantine',
    'Tiberios III':                     'Tiberius III',
    'Leo III the Isaurian':             'Leo III the Isaurian',
    'Constantine V Kopronymos':         'Constantine V',
    'Leo IV the Khazar':                'Leo IV the Khazar',
    'Nikephoros I':                     'Nikephoros I',
    'Michael I Rhangabe':               'Michael I Rhangabe',
    'Leo V the Armenian':               'Leo V the Armenian',
    'Michael II the Amorian':           'Michael II',
    'Michael III the Drunkard':         'Michael III',
    'Basil I the Macedonian':           'Basil I',
    'Leo VI the Wise':                  'Leo VI the Wise',
    'Constantine VII Porphyrogennetos': 'Constantine VII',
    'Romanos I Lekapenos':              'Romanos I Lekapenos',
    'Nikephoros II Phokas':             'Nikephoros II Phokas',
    'John I Tzimiskes':                 'John I Tzimiskes',
    'Basil II the Bulgar-Slayer':       'Basil II',
    'Romanos III Argyros':              'Romanos III',
    'Michael IV the Paphlagonian':      'Michael IV',
    'Michael V Kalaphates':             'Michael V',
    'Zoe (with Theodora)':              'Zoe (empress)',
    'Constantine IX Monomachos':        'Constantine IX',
    'Isaac I Komnenos':                 'Isaac I Komnenos',
    'Constantine X Doukas':             'Constantine X',
    'Romanos IV Diogenes':              'Romanos IV Diogenes',
    'Michael VII Doukas':               'Michael VII Doukas',
    'Nikephoros III Botaneiates':       'Nikephoros III',
    'Alexios I Komnenos':               'Alexios I Komnenos',
    'John II Komnenos':                 'John II Komnenos',
    'Manuel I Komnenos':                'Manuel I Komnenos',
    'Alexios II Komnenos':              'Alexios II Komnenos',
    'Andronikos I Komnenos':            'Andronikos I Komnenos',
    'Isaac II Angelos':                 'Isaac II Angelos',
    'Alexios III Angelos':              'Alexios III Angelos',
    'Alexios IV Angelos':               'Alexios IV Angelos',
    'Alexios V Mourtzouphlos':          'Alexios V Doukas',
    'Theodore I Laskaris':              'Theodore I Laskaris',
    'John III Doukas Vatatzes':         'John III Doukas Vatatzes',
    'Theodore II Laskaris':             'Theodore II Laskaris',
    'John IV Laskaris':                 'John IV Laskaris',
    'Michael VIII Palaiologos':         'Michael VIII Palaiologos',
    'Andronikos II Palaiologos':        'Andronikos II Palaiologos',
    'Michael IX Palaiologos':           'Michael IX Palaiologos',
    'Andronikos III Palaiologos':       'Andronikos III Palaiologos',
    'John V Palaiologos':               'John V Palaiologos',
    'John VI Kantakouzenos':            'John VI Kantakouzenos',
    'Andronikos IV Palaiologos':        'Andronikos IV Palaiologos',
    'John VII Palaiologos':             'John VII Palaiologos',
    'Manuel II Palaiologos':            'Manuel II Palaiologos',
    'John VIII Palaiologos':            'John VIII Palaiologos',
    'Constantine XI Palaiologos Dragases': 'Constantine XI',
}

def get_page_length(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "size",
        "format": "json"
    }
    try:
        r = session.get(url, params=params, timeout=10)
        pages = r.json()['query']['pages']
        for page in pages.values():
            return page['revisions'][0]['size']
    except:
        return 0

def get_page_views(title):
    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/{title.replace(' ','_')}/daily/20250101/20250131"
    try:
        r = session.get(url, timeout=10)
        items = r.json().get('items', [])
        if items:
            return int(sum(i['views'] for i in items) / len(items))
        return 0
    except:
        return 0

print("Fetching Wikipedia metrics...\n")
print(f"{'#':<4} {'Emperor':<45} {'Length':>8} {'Views':>8}")
print("-" * 70)

lengths = []
views = []

for _, row in df.iterrows():
    name = row['Simple_Name_With_Epithets']
    wiki_title = WIKI_TITLES.get(name, name)
    
    length = get_page_length(wiki_title)
    view   = get_page_views(wiki_title)
    
    lengths.append(length)
    views.append(view)
    
    print(f"{int(row['List_Number']):<4} {name:<45} {length:>8,} {view:>8,}")
    time.sleep(0.8)

df['Wiki_Length'] = lengths
df['Wiki_Views']  = views
df.to_csv('ERE_List_20260226.csv', index=False)

print("\nDone! Wiki metrics saved to CSV.")
print("\nTop 15 by combined rank:")

df['Length_Rank'] = df['Wiki_Length'].rank(ascending=False)
df['Views_Rank']  = df['Wiki_Views'].rank(ascending=False)
df['Wiki_Score']  = (df['Length_Rank'] + df['Views_Rank']) / 2
top15 = df.nsmallest(15, 'Wiki_Score')[['Simple_Name_With_Epithets', 'Wiki_Length', 'Wiki_Views', 'Wiki_Score']]
print(top15.to_string(index=False))
df.to_csv('ERE_List_20260226.csv', index=False)

Fetching Wikipedia metrics...

#    Emperor                                         Length    Views
----------------------------------------------------------------------
1    Constantine I the Great                        183,238    4,576
2    Constantius II                                  68,765      716
3    Julian the Apostate                                 58      128
4    Jovian                                             909       35
5    Valens                                          47,594      477
6    Theodosios I the Great                         106,509    1,489
7    Arkadios                                             0        0
8    Theodosios II                                   32,073      490
9    Markianos                                       60,908        0
10   Leo I the Thracian                                  76       32
11   Leo II                                             776        3
12   Zeno                                             2,519       54
1

### 3a - Fix Problem Entries

Run after the main Wikipedia fetch.

In [11]:
import pandas as pd
import requests
import time

df = pd.read_csv('ERE_List_20260226.csv')

session = requests.Session()
session.headers.update({
    'User-Agent': 'ByzantineEmperorsTool/1.0 (educational project; you@example.com)'
})

# Fixed title mappings for problem entries
FIXES = {
    'Julian the Apostate':            'Julian (emperor)',
    'Arkadios':                       'Arcadius',
    'Leo I the Thracian':             'Leo I (emperor)',
    'Maurikios':                      'Maurice (emperor)',
    'Phokas':                         'Phocas',
    'Leontios':                       'Leontius (emperor)',
    'Theophilos':                     'Theophilos (emperor)',
    'Eirene':                         'Irene of Athens',
    'Constantine XI Palaiologos Dragases': 'Constantine XI',
    'Romanos III Argyros':            'Romanos III',
    'Zoe (with Theodora)':            'Zoe (empress)',
    'Constantine IX Monomachos':      'Constantine IX',
    'Michael I Rhangabe':             'Michael I of Bulgaria',
    'Anastasios II':                  'Anastasius II',
    'Theodosios III':                 'Theodosius III',
    'Basiliskos':                     'Basiliscus',
    'Philippikos':                    'Philippicus',
    'Constantine X Doukas':           'Constantine X Doukas',
    'Nikephoros III Botaneiates':     'Nikephoros III Botaneiates',
    'Michael IV the Paphlagonian':    'Michael IV the Paphlagonian',
    'Michael V Kalaphates':           'Michael V',
    'Michael VI Bringas':             'Michael VI Bringas',
    'Constantine III':                'Constantine III (Byzantine emperor)',
    'John IV Laskaris':               'John IV Laskaris',
}

def get_page_length(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "revisions",
        "rvprop": "size",
        "format": "json"
    }
    try:
        r = session.get(url, params=params, timeout=10)
        pages = r.json()['query']['pages']
        for page in pages.values():
            return page['revisions'][0]['size']
    except:
        return 0

def get_page_views(title):
    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/{title.replace(' ','_')}/daily/20250101/20250131"
    try:
        r = session.get(url, timeout=10)
        items = r.json().get('items', [])
        if items:
            return int(sum(i['views'] for i in items) / len(items))
        return 0
    except:
        return 0

print("Fixing problem entries...\n")
print(f"{'Emperor':<45} {'Length':>8} {'Views':>8}")
print("-" * 65)

for _, row in df.iterrows():
    name = row['Simple_Name_With_Epithets']
    if name not in FIXES:
        continue
    
    wiki_title = FIXES[name]
    length = get_page_length(wiki_title)
    views  = get_page_views(wiki_title)
    
    df.loc[df['Simple_Name_With_Epithets'] == name, 'Wiki_Length'] = length
    df.loc[df['Simple_Name_With_Epithets'] == name, 'Wiki_Views']  = views
    
    print(f"{name:<45} {length:>8,} {views:>8,}")
    time.sleep(0.8)

df.to_csv('ERE_List_20260226.csv', index=False)

# Recalculate top 15
df['Length_Rank'] = df['Wiki_Length'].rank(ascending=False)
df['Views_Rank']  = df['Wiki_Views'].rank(ascending=False)
df['Wiki_Score']  = (df['Length_Rank'] + df['Views_Rank']) / 2
top15 = df.nsmallest(15, 'Wiki_Score')[['Simple_Name_With_Epithets', 'Wiki_Length', 'Wiki_Views', 'Wiki_Score']]

print("\nDone! Updated top 15:")
print(top15.to_string(index=False))
df.to_csv('ERE_List_20260226.csv', index=False)

Fixing problem entries...

Emperor                                         Length    Views
-----------------------------------------------------------------
Julian the Apostate                            109,881    1,170
Arkadios                                        33,071      398
Leo I the Thracian                              21,828      332
Basiliskos                                      55,447      227
Maurikios                                       47,240      384
Phokas                                          23,669      356
Constantine III                                    110       55
Leontios                                            22        1
Philippikos                                      9,990        0
Anastasios II                                      345        4
Theodosios III                                  21,522      135
Eirene                                          49,148      442
Michael I Rhangabe                                   0        0
Theophilos 

## 4 - Download Emperor Images

Skips already-downloaded files. Safe to re-run.

In [12]:
import pandas as pd
import requests
import os
import time
from urllib.parse import urlparse

df = pd.read_csv('ERE_List_20260226.csv')

# Create images folder
os.makedirs('images', exist_ok=True)

session = requests.Session()
session.headers.update({
    'User-Agent': 'ByzantineEmperorsTool/1.0 (educational project; you@example.com)'
})

print("Downloading images...\n")

local_urls = []
failed = 0

for idx, row in df.iterrows():
    name = row['Simple_Name_With_Epithets']
    img_url = str(row['Image_URL'])
    num = int(row['List_Number'])

    if img_url in ('PLACEHOLDER', 'NOT_FOUND', 'NO_IMAGE', 'ERROR', 'nan'):
        print(f"  [{num}] {name} — no URL, skipping")
        local_urls.append(img_url)
        continue

    # Get file extension from URL
    parsed = urlparse(img_url)
    ext = os.path.splitext(parsed.path)[-1].lower()
    if ext not in ('.jpg', '.jpeg', '.png', '.gif', '.webp'):
        ext = '.jpg'

    filename = f"images/{num:03d}_{ext}"

    if os.path.exists(filename):
        print(f"  [{num}] {name} — already exists, skipping")
        local_urls.append(filename)
        continue

    try:
        resp = session.get(img_url, timeout=15)
        if resp.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(resp.content)
            print(f"  [{num}] {name} — saved as {filename}")
            local_urls.append(filename)
        else:
            print(f"  [{num}] {name} — HTTP {resp.status_code}, keeping URL")
            local_urls.append(img_url)
            failed += 1
    except Exception as e:
        print(f"  [{num}] {name} — ERROR: {e}")
        local_urls.append(img_url)
        failed += 1

    time.sleep(0.3)

# Save local paths back to CSV
df['Local_Image'] = local_urls
df.to_csv('ERE_List_20260226.csv', index=False)

print(f"\nDone! {len(df) - failed} images downloaded.")
print(f"Failed: {failed}")
print("Local paths saved to 'Local_Image' column in CSV.")


  [1] Constantine I the Great — already exists, skipping
  [2] Constantius II — already exists, skipping
  [3] Julian the Apostate — already exists, skipping
  [4] Jovian — already exists, skipping
  [5] Valens — already exists, skipping
  [6] Theodosios I the Great — already exists, skipping
  [7] Arkadios — already exists, skipping
  [8] Theodosios II — already exists, skipping
  [9] Markianos — already exists, skipping
  [10] Leo I the Thracian — already exists, skipping
  [11] Leo II — already exists, skipping
  [12] Zeno — already exists, skipping
  [13] Basiliskos — already exists, skipping
  [14] Anastasios I — already exists, skipping
  [15] Justin I — already exists, skipping
  [16] Justinian I the Great — already exists, skipping
  [17] Justin II — already exists, skipping
  [18] Tiberios II Constantine — already exists, skipping
  [19] Maurikios — already exists, skipping
  [20] Phokas — already exists, skipping
  [21] Herakleios — already exists, skipping
  [22] Constantin

## 5 - Generate Scatter Chart

Writes byzantine_scatter_v3.html. Emperors with blank D/E appear in the Unrankable box.

In [7]:
import pandas as pd
import json
import base64
import os
import math

df = pd.read_csv('ERE_List_20260226.csv')

# Calculate composite score
df['Composite'] = ((df['Effectiveness'] * 0.6 + df['Difficulty'] * 0.4) * 5).round(1)

# Notable emperors to always label
NOTABLE = {
    'Constantine I the Great',
    'Julian the Apostate',
    'Theodosios I the Great',
    'Justinian I the Great',
    'Herakleios',
    'Basil II the Bulgar-Slayer',
    'Alexios I Komnenos',
    'Constantine VII Porphyrogennetos',
    'John II Komnenos',
    'Manuel I Komnenos',
    'Leo III the Isaurian',
    'Andronikos I Komnenos',
    'Constantine XI Palaiologos Dragases',
    'Eirene',
    'Valens',
}

def img_to_base64(path):
    if not path or str(path) in ('PLACEHOLDER', 'NOT_FOUND', 'NO_IMAGE', 'ERROR', 'nan'):
        return ''
    if not os.path.exists(str(path)):
        return ''
    ext = os.path.splitext(str(path))[-1].lower().replace('.', '')
    if ext == 'jpg': ext = 'jpeg'
    try:
        with open(str(path), 'rb') as f:
            data = base64.b64encode(f.read()).decode('utf-8')
        return f"data:image/{ext};base64,{data}"
    except:
        return ''

def load_deco(path):
    if not os.path.exists(path):
        return ''
    ext = os.path.splitext(path)[-1].lower().replace('.', '')
    if ext == 'jpg': ext = 'jpeg'
    try:
        with open(path, 'rb') as f:
            data = base64.b64encode(f.read()).decode('utf-8')
        return f"data:image/{ext};base64,{data}"
    except:
        return ''

arch_b64   = load_deco('constantinearch.png')
column_b64 = load_deco('Gurlitt_Justinian_column-removebg-preview.png')

emperors = []
unranked = []

for _, row in df.iterrows():
    name = str(row['Simple_Name_With_Epithets'])
    d = row['Difficulty']
    e = row['Effectiveness']
    img = img_to_base64(str(row.get('Local_Image', '')))
    is_blank = (isinstance(d, float) and math.isnan(d)) or (isinstance(e, float) and math.isnan(e))

    if not is_blank:
        emperors.append({
            "num":        int(row['List_Number']),
            "name":       name,
            "dynasty":    str(row['Broad_Dynasty']),
            "start":      int(row['Reign_Start']),
            "end":        int(row['Reign_End']),
            "years":      int(row['Reign_Length_Years']),
            "difficulty": int(d),
            "effective":  int(e),
            "composite":  float(row['Composite']),
            "summary":    str(row['Summary']),
            "image":      img,
            "label":      'notable' if name in NOTABLE else 'none',
        })
        print(f"  Ranked   #{int(row['List_Number'])}: {name}")
    else:
        unranked.append({
            "num":     int(row['List_Number']),
            "name":    name,
            "dynasty": str(row['Broad_Dynasty']),
            "start":   int(row['Reign_Start']),
            "end":     int(row['Reign_End']),
            "summary": str(row['Summary']),
            "image":   img,
        })
        print(f"  Unranked #{int(row['List_Number'])}: {name}")

data_json     = json.dumps(emperors)
unranked_json = json.dumps(unranked)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Eastern Roman Emperors – Difficulty vs Effectiveness</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ background: #f8f5ef; color: #222; font-family: 'Georgia', serif; }}
  h1 {{ text-align: center; color: #8b1a1a; font-size: 1.9em; padding: 24px 20px 4px; letter-spacing: 1px; }}
  .subtitle {{ text-align: center; color: #666; font-size: 0.85em; padding-bottom: 16px; }}
  #chart-outer {{ width: 100%; overflow: hidden; }}
  #chart-wrap {{ position: relative; width: 1280px; height: 720px; margin: 0 auto; transform-origin: top center; }}
  .info-btn {{ display:inline-block; width:15px; height:15px; line-height:15px; text-align:center;
    background:#8b1a1a; color:#fff; border-radius:50%; font-size:10px; cursor:pointer;
    vertical-align:middle; margin-left:8px; font-family:Georgia,serif; font-style:italic; }}
  #info-tooltip {{ position:fixed; background:rgba(255,252,245,0.98); border:1px solid #8b1a1a;
    border-radius:6px; padding:12px 14px; display:none; max-width:360px; z-index:200;
    box-shadow:2px 2px 8px rgba(0,0,0,0.15); font-family:Georgia,serif; font-size:0.82em; color:#444;
    line-height:1.5; }}
  canvas {{ position: absolute; top: 0; left: 0; }}
  #tooltip {{
    position: fixed; background: rgba(255,252,245,0.98); border: 1px solid #8b1a1a;
    border-radius: 6px; padding: 12px; pointer-events: none; display: none;
    max-width: 290px; z-index: 100; box-shadow: 2px 2px 8px rgba(0,0,0,0.15);
  }}
  #tooltip img {{ width: 75px; height: 95px; object-fit: cover; border-radius: 4px; float: left; margin-right: 10px; }}
  #tooltip .tip-name    {{ color: #8b1a1a; font-size: 0.95em; font-weight: bold; margin-bottom: 3px; }}
  #tooltip .tip-dynasty {{ color: #888; font-size: 0.78em; margin-bottom: 3px; }}
  #tooltip .tip-reign   {{ color: #666; font-size: 0.76em; margin-bottom: 5px; }}
  #tooltip .tip-scores  {{ color: #222; font-size: 0.8em; margin-bottom: 5px; }}
  #tooltip .tip-summary {{ color: #555; font-size: 0.73em; line-height: 1.4; clear: both; margin-top: 8px; border-top: 1px solid #ddd; padding-top: 6px; }}
  #legend {{
    display: flex; justify-content: center; gap: 12px; padding: 10px 40px;
    flex-wrap: wrap; font-size: 0.75em; color: #555; max-width: 560px; margin: 0 auto;
  }}
  .legend-item {{ display: flex; align-items: center; gap: 6px; }}
  .legend-dot  {{ width: 12px; height: 12px; border-radius: 50%; }}
  #key {{ text-align: center; font-size: 0.75em; color: #888; padding-bottom: 20px; }}
</style>
</head>
<body>

<h1>Eastern Roman Emperors <span class="info-btn" id="info-btn">i</span></h1>
<div id="info-tooltip">Any crowned Emperor of the Roman Empire (or close enough) from the foundation of Constantinople (Constantine I) to Constantine XI. Excluding any Emperor who primarily ruled in the West. Unrankable group in bottom left.</div>
<p class="subtitle">Difficulty of Reign vs Effectiveness &nbsp;·&nbsp; Circle size = length of reign &nbsp;·&nbsp; Hover for details</p>

<div id="chart-outer">
<div id="chart-wrap">
  <canvas id="bg" width="1280" height="720"></canvas>
  <canvas id="fg" width="1280" height="720"></canvas>
</div>
</div>

<div id="legend">
<div class="legend-item" data-dyn="Constantinian" style="cursor:pointer"><div class="legend-dot" style="background:#7b2d8b"></div>Constantinian</div>
  <div class="legend-item" data-dyn="Valentinianic" style="cursor:pointer"><div class="legend-dot" style="background:#9b3dab"></div>Valentinianic</div>
  <div class="legend-item" data-dyn="Theodosian" style="cursor:pointer"><div class="legend-dot" style="background:#4a90d9"></div>Theodosian</div>
  <div class="legend-item" data-dyn="Leonid" style="cursor:pointer"><div class="legend-dot" style="background:#2e6da4"></div>Leonid</div>
  <div class="legend-item" data-dyn="Justinianic" style="cursor:pointer"><div class="legend-dot" style="background:#c0392b"></div>Justinianic</div>
  <div class="legend-item" data-dyn="Heraklean" style="cursor:pointer"><div class="legend-dot" style="background:#e67e22"></div>Heraklean</div>
  <div class="legend-item" data-dyn="Isaurian" style="cursor:pointer"><div class="legend-dot" style="background:#27ae60"></div>Isaurian</div>
  <div class="legend-item" data-dyn="Amorian" style="cursor:pointer"><div class="legend-dot" style="background:#16a085"></div>Amorian</div>
  <div class="legend-item" data-dyn="Macedonian" style="cursor:pointer"><div class="legend-dot" style="background:#d4a017"></div>Macedonian</div>
  <div class="legend-item" data-dyn="Komnenian" style="cursor:pointer"><div class="legend-dot" style="background:#8b1a1a"></div>Komnenian</div>
  <div class="legend-item" data-dyn="Doukas" style="cursor:pointer"><div class="legend-dot" style="background:#b7950b"></div>Doukas</div>
  <div class="legend-item" data-dyn="Angelid" style="cursor:pointer"><div class="legend-dot" style="background:#717d7e"></div>Angelid</div>
  <div class="legend-item" data-dyn="Laskarid" style="cursor:pointer"><div class="legend-dot" style="background:#1a5276"></div>Laskarid</div>
  <div class="legend-item" data-dyn="Palaiologan" style="cursor:pointer"><div class="legend-dot" style="background:#784212"></div>Palaiologan</div>
  <div class="legend-item" data-dyn="Paphlagonian" style="cursor:pointer"><div class="legend-dot" style="background:#5d6d7e"></div>Paphlagonian</div>
  <div class="legend-item" data-dyn="Non-dynastic" style="cursor:pointer"><div class="legend-dot" style="background:#bbb"></div>Non-dynastic</div>
<div class="legend-item" id="legend-show-all" style="cursor:pointer;font-weight:bold;color:#8b1a1a;border:1px solid #8b1a1a;border-radius:4px;padding:1px 8px;">Show all</div>
  <div class="legend-item" id="legend-hide-all" style="cursor:pointer;font-weight:bold;color:#666;border:1px solid #999;border-radius:4px;padding:1px 8px;">Hide all</div>
</div>
<p id="key">Circle size reflects length of reign · Name shown for notable emperors</p>

<div id="tooltip"></div>

<script>
const DATA = {data_json};
const W = 1280, H = 720;
const PAD = {{left:55, right:100, top:75, bottom:80}};
const plotW = W - PAD.left - PAD.right;
const plotH = H - PAD.top - PAD.bottom;
const MIN_R = 14, MAX_R = 52;
const maxYears = Math.max(...DATA.map(e => e.years));

function sx(d) {{ return PAD.left + ((d-1)/19)*plotW; }}
function sy(e) {{ return PAD.top + plotH - ((e-1)/19)*plotH; }}
function radius(y) {{ return MIN_R + (y/maxYears)*(MAX_R-MIN_R); }}

const DYNASTY_COLORS = {{
  'Constantinian':  '#7b2d8b',
  'Valentinianic':  '#9b3dab',
  'Theodosian':     '#4a90d9',
  'Leonid':         '#2e6da4',
  'Justinianic':    '#c0392b',
  'Heraklean':      '#e67e22',
  'Isaurian':       '#27ae60',
  'Amorian':        '#16a085',
  'Macedonian':     '#d4a017',
  'Komnenian':      '#8b1a1a',
  'Doukas':         '#b7950b',
  'Angelid':        '#717d7e',
  'Laskarid':       '#1a5276',
  'Palaiologan':    '#784212',
  'Paphlagonian':   '#5d6d7e',
  'N/A':            '#999',
  'Non-dynastic':   '#bbb',
  '':               '#bbb',
}};

function ringColor(dynasty) {{
  return DYNASTY_COLORS[dynasty] || '#2c7bb6';
}}

const bg = document.getElementById('bg').getContext('2d');

// Column drawn first so chart renders on top
const colImg = new Image();
colImg.onload = function() {{
  bg.globalAlpha = 0.4;
  const colH = H * 0.4;
  const colW = colH / 4;
  const colY = PAD.top + (plotH / 2) - (colH / 2);
  bg.drawImage(colImg, PAD.left + plotW + 5, colY, colW, colH);
  bg.globalAlpha = 1.0;
}};
if('{column_b64}') colImg.src = `{column_b64}`;

// Quadrant shading
const mx = sx(10), my = sy(10);
bg.fillStyle='rgba(255,210,0,0.08)';   bg.fillRect(PAD.left,PAD.top,mx-PAD.left,my-PAD.top);
bg.fillStyle='rgba(0,160,0,0.08)';     bg.fillRect(mx,PAD.top,PAD.left+plotW-mx,my-PAD.top);
bg.fillStyle='rgba(220,100,100,0.08)'; bg.fillRect(PAD.left,my,mx-PAD.left,PAD.top+plotH-my);
bg.fillStyle='rgba(150,150,150,0.07)'; bg.fillRect(mx,my,PAD.left+plotW-mx,PAD.top+plotH-my);

// Grid
bg.strokeStyle='#e0e0e0'; bg.lineWidth=1;
for(let i=1;i<=20;i++){{
  bg.beginPath(); bg.moveTo(sx(i),PAD.top); bg.lineTo(sx(i),PAD.top+plotH); bg.stroke();
  bg.beginPath(); bg.moveTo(PAD.left,sy(i)); bg.lineTo(PAD.left+plotW,sy(i)); bg.stroke();
}}

// Quadrant dividers
bg.strokeStyle='#aaa'; bg.lineWidth=1.5; bg.setLineDash([5,4]);
bg.beginPath(); bg.moveTo(mx,PAD.top); bg.lineTo(mx,PAD.top+plotH); bg.stroke();
bg.beginPath(); bg.moveTo(PAD.left,my); bg.lineTo(PAD.left+plotW,my); bg.stroke();
bg.setLineDash([]);

// Border
bg.strokeStyle='#999'; bg.lineWidth=2;
bg.strokeRect(PAD.left, PAD.top, plotW, plotH);

// Tick labels
bg.fillStyle='#999'; bg.font='11px Georgia'; bg.textAlign='center';
for(let i=2;i<=20;i+=2) bg.fillText(i, sx(i), PAD.top+plotH+18);
bg.textAlign='right';
for(let i=2;i<=20;i+=2) bg.fillText(i, PAD.left-8, sy(i)+4);

// Axis titles
bg.fillStyle='#666'; bg.font='13px Georgia'; bg.textAlign='center';
bg.fillText('Difficulty of Reign →', PAD.left+plotW/2, H-28);
bg.font='11px Georgia'; bg.fillStyle='#999';
bg.fillText('Defined by how challenging was the state of the empire at ascension to the throne', PAD.left+plotW/2, H-12);
bg.save(); bg.translate(16,PAD.top+plotH/2); bg.rotate(-Math.PI/2);
bg.fillStyle='#666'; bg.font='13px Georgia'; bg.textAlign='center';
bg.fillText('Effectiveness →', 0, 0); bg.restore();

// Quadrant labels
bg.font='italic 10px Georgia'; bg.fillStyle='rgba(0,0,0,0.18)'; bg.textAlign='center';
bg.fillText('Easy + Effective', PAD.left+plotW*0.25, PAD.top+16);
bg.fillText('Hard + Effective', PAD.left+plotW*0.75, PAD.top+16);
bg.fillText('Easy + Ineffective', PAD.left+plotW*0.25, PAD.top+plotH-8);
bg.fillText('Hard + Ineffective', PAD.left+plotW*0.75, PAD.top+plotH-8);

// Watermark
bg.fillStyle='rgba(0,0,0,0.25)';
bg.font='italic 12px Georgia'; bg.textAlign='right';
bg.fillText('Eric Cook, @ecocinar', PAD.left+plotW-8, H-2);

// Arch drawn after chart so it sits on top
const archImg = new Image();
archImg.onload = function() {{
  bg.globalAlpha = 0.5;
  bg.drawImage(archImg, 5, 5, 220, 60);
  bg.globalAlpha = 1.0;
}};
if('{arch_b64}') archImg.src = `{arch_b64}`;

// Compute positions first
const fg = document.getElementById('fg').getContext('2d');
const positions = [];

DATA.forEach(emp => {{
  positions.push({{
    cx: sx(emp.difficulty),
    cy: sy(emp.effective),
    r:  radius(emp.years),
    homeX: sx(emp.difficulty),
    homeY: sy(emp.effective),
    emp
  }});
}});

// Collision resolution: push overlapping portraits apart,
// gently pulling back toward true score position
(function resolveCollisions() {{
  const ITER = 40, PADDING = 2, PULL = 0.08;
  for(let it = 0; it < ITER; it++) {{
    for(let i = 0; i < positions.length; i++) {{
      for(let j = i + 1; j < positions.length; j++) {{
        const a = positions[i], b = positions[j];
        let dx = b.cx - a.cx, dy = b.cy - a.cy;
        let dist = Math.sqrt(dx*dx + dy*dy);
        const minDist = a.r + b.r + PADDING;
        if(dist < minDist) {{
          if(dist < 0.01) {{ dx = Math.random()-0.5; dy = Math.random()-0.5; dist = 0.01; }}
          const push = (minDist - dist) / 2;
          const ux = dx/dist, uy = dy/dist;
          a.cx -= ux*push; a.cy -= uy*push;
          b.cx += ux*push; b.cy += uy*push;
        }}
      }}
    }}
    positions.forEach(p => {{
      p.cx += (p.homeX - p.cx) * PULL;
      p.cy += (p.homeY - p.cy) * PULL;
      p.cx = Math.max(PAD.left + p.r, Math.min(PAD.left + plotW - p.r, p.cx));
      p.cy = Math.max(PAD.top + p.r, Math.min(PAD.top + plotH - p.r, p.cy));
    }});
  }}
}})();

// Dynasty visibility + preloaded redraw
const hiddenDynasties = new Set();

positions.forEach(p => {{
  if(p.emp.image) {{
    p.img = new Image();
    p.img.onload = () => drawAll();
    p.img.src = p.emp.image;
  }}
}});

function drawAll() {{
  fg.clearRect(0, 0, W, H);
  positions.forEach(p => {{
    if(hiddenDynasties.has(p.emp.dynasty)) return;
    const cx = p.cx, cy = p.cy, r = p.r, emp = p.emp;

    fg.beginPath(); fg.arc(cx,cy,r+3,0,Math.PI*2);
    fg.fillStyle = ringColor(emp.dynasty); fg.fill();

    if(p.img && p.img.complete && p.img.naturalWidth > 0) {{
      fg.save();
      fg.beginPath(); fg.arc(cx,cy,r,0,Math.PI*2); fg.clip();
      fg.drawImage(p.img, cx-r, cy-r, r*2, r*2);
      fg.restore();
      if(emp.label === 'notable') {{
        const shortName = emp.name.split(' ').slice(0,2).join(' ');
        const fontSize = Math.max(8, Math.min(11, r*0.35));
        fg.save();
        fg.beginPath(); fg.arc(cx,cy,r,0,Math.PI*2); fg.clip();
        fg.fillStyle='rgba(0,0,0,0.45)'; fg.fillRect(cx-r,cy-r,r*2,r*2);
        fg.restore();
        fg.fillStyle='#fff';
        fg.font=`bold ${{fontSize}}px Georgia`;
        fg.textAlign='center';
        fg.fillText(shortName, cx, cy + r*0.5);
      }}
    }} else {{
      fg.beginPath(); fg.arc(cx,cy,r,0,Math.PI*2);
      fg.fillStyle='#e8e0d0'; fg.fill();
      fg.fillStyle='#8b1a1a';
      fg.font=`bold ${{Math.max(9,r*0.4)}}px Georgia`;
      fg.textAlign='center';
      fg.fillText(emp.name.charAt(0), cx, cy+4);
    }}
  }});
}}
drawAll();

// Legend click toggles
document.querySelectorAll('.legend-item[data-dyn]').forEach(item => {{
  item.addEventListener('click', () => {{
    const dyn = item.getAttribute('data-dyn');
    if(hiddenDynasties.has(dyn)) {{
      hiddenDynasties.delete(dyn);
      item.style.opacity = '1';
    }} else {{
      hiddenDynasties.add(dyn);
      item.style.opacity = '0.3';
    }}
    drawAll();
  }});
}});

// Show all / Hide all buttons
document.getElementById('legend-show-all').addEventListener('click', () => {{
  hiddenDynasties.clear();
  document.querySelectorAll('.legend-item[data-dyn]').forEach(i => i.style.opacity = '1');
  drawAll();
}});
document.getElementById('legend-hide-all').addEventListener('click', () => {{
  document.querySelectorAll('.legend-item[data-dyn]').forEach(i => {{
    hiddenDynasties.add(i.getAttribute('data-dyn'));
    i.style.opacity = '0.3';
  }});
  drawAll();
}});

// Tooltip
const tip = document.getElementById('tooltip');
document.getElementById('fg').addEventListener('mousemove', function(e) {{
  const rect = this.getBoundingClientRect();
  const scale = rect.width / 1280;
  const mx = (e.clientX-rect.left) / scale, my = (e.clientY-rect.top) / scale;
  let found = null;
  for(const p of [...positions].sort((a,b) => a.r-b.r)) {{
  if(hiddenDynasties.has(p.emp.dynasty)) continue;
    const dx=mx-p.cx, dy=my-p.cy;
    if(Math.sqrt(dx*dx+dy*dy) < p.r+4) {{ found=p; break; }}
  }}
  if(found) {{
    const e2 = found.emp;
    const imgTag = e2.image ? `<img src="${{e2.image}}" onerror="this.style.display='none'">` : '';
    tip.innerHTML = `
      ${{imgTag}}
      <div class="tip-name">${{e2.name}}</div>
      <div class="tip-dynasty">${{e2.dynasty}}</div>
      <div class="tip-reign">${{e2.start}} – ${{e2.end}} (${{e2.years}} years)</div>
      <div class="tip-scores">Difficulty: ${{e2.difficulty}} &nbsp;|&nbsp; Effectiveness: ${{e2.effective}} &nbsp;|&nbsp; Score: ${{e2.composite}}</div>
      <div class="tip-summary">${{e2.summary}}</div>
    `;
    tip.style.display='block';
    let tx=e.clientX+15, ty=e.clientY-10;
    if(tx+300>window.innerWidth) tx=e.clientX-305;
    if(ty+320>window.innerHeight) ty=e.clientY-320;
    tip.style.left=tx+'px'; tip.style.top=ty+'px';
    this.style.cursor='pointer';
  }} else {{
    tip.style.display='none';
    this.style.cursor='default';
  }}
}});
document.getElementById('fg').addEventListener('mouseleave', ()=>tip.style.display='none');
</script>
<div style="max-width:1280px;margin:6px auto 16px auto;padding:0 20px;">
<div style="display:inline-block;background:rgba(255,252,245,0.9);border:1px solid #8b1a1a;border-radius:6px;padding:8px 14px;font-family:Georgia,serif;">
<div style="color:#8b1a1a;font-size:0.78em;font-weight:bold;margin-bottom:6px;letter-spacing:0.5px;">&#9670; Unrankable</div>
<div id="unrankable-list" style="display:flex;flex-wrap:wrap;gap:10px;align-items:center;"></div>
</div>
</div>

<script>
(function() {{
  var UNRANKED = UNRANKED_DATA;
  var urTip = document.getElementById("tooltip");
  var ul = document.getElementById("unrankable-list");
  UNRANKED.forEach(function(emp) {{
    var item = document.createElement("div");
    item.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:3px;cursor:pointer;";
    if(emp.image) {{
      var img = document.createElement("img");
      img.src = emp.image;
      img.style.cssText = "width:30px;height:37px;object-fit:cover;border-radius:3px;border:1px solid #ccc;opacity:0.8;";
      item.appendChild(img);
    }}
    var lbl = document.createElement("span");
    lbl.textContent = emp.num + ". " + emp.name.split(" ")[0];
    lbl.style.cssText = "font-size:0.62em;color:#888;font-family:Georgia,serif;";
    item.appendChild(lbl);
    item.addEventListener("mouseenter", function(ev) {{
      var imgTag = emp.image ? "<img src='" + emp.image + "' onerror='this.style.display=none'>" : "";
      urTip.innerHTML = imgTag
        + "<div class=tip-name>" + emp.name + "</div>"
        + "<div class=tip-dynasty>" + emp.dynasty + "</div>"
        + "<div class=tip-reign>" + emp.start + " - " + emp.end + "</div>"
        + "<div class=tip-scores style=color:#8b1a1a;font-style:italic>Unrankable</div>"
        + "<div class=tip-summary>" + emp.summary + "</div>";
      urTip.style.display = "block";
      var tx = ev.clientX + 15, ty = ev.clientY - 10;
      if(tx + 300 > window.innerWidth) tx = ev.clientX - 305;
      if(ty + 280 > window.innerHeight) ty = ev.clientY - 280;
      urTip.style.left = tx + "px"; urTip.style.top = ty + "px";
    }});
    item.addEventListener("mouseleave", function() {{ urTip.style.display = "none"; }});
    ul.appendChild(item);
  }});

  var infoBtn = document.getElementById("info-btn");
  var infoTip = document.getElementById("info-tooltip");
  if(infoBtn) {{
    infoBtn.addEventListener("mouseenter", function(ev) {{
      infoTip.style.display = "block";
      var tx = ev.clientX + 15, ty = ev.clientY + 10;
      if(tx + 380 > window.innerWidth) tx = ev.clientX - 375;
      infoTip.style.left = tx + "px"; infoTip.style.top = ty + "px";
    }});
    infoBtn.addEventListener("mouseleave", function() {{ infoTip.style.display = "none"; }});
  }}
}})();
</script>

<script>
function scaleChart() {{
  var wrap = document.getElementById("chart-wrap");
  var outer = document.getElementById("chart-outer");
  var available = outer.clientWidth - 8;
  var scale = Math.min(1, available / 1280);
  wrap.style.transform = "scale(" + scale + ")";
  outer.style.height = (720 * scale) + "px";
}}
scaleChart();
window.addEventListener("resize", scaleChart);
</script>
</body>
</html>"""

html = html.replace('UNRANKED_DATA', unranked_json)
with open('byzantine_scatter_v3.html', 'w', encoding='utf-8') as f:
    f.write(html)

print("\nDone! Open byzantine_scatter_v3.html in your browser.")

  Ranked   #1: Constantine I the Great
  Ranked   #2: Constantius II
  Ranked   #3: Julian the Apostate
  Ranked   #4: Jovian
  Ranked   #5: Valens
  Ranked   #6: Theodosios I the Great
  Ranked   #7: Arkadios
  Ranked   #8: Theodosios II
  Ranked   #9: Markianos
  Ranked   #10: Leo I the Thracian
  Unranked #11: Leo II
  Ranked   #12: Zeno
  Ranked   #13: Basiliskos
  Ranked   #14: Anastasios I
  Ranked   #15: Justin I
  Ranked   #16: Justinian I the Great
  Ranked   #17: Justin II
  Ranked   #18: Tiberios II Constantine
  Ranked   #19: Maurikios
  Ranked   #20: Phokas
  Ranked   #21: Herakleios
  Ranked   #22: Constantine III
  Unranked #23: Heraklonas
  Ranked   #24: Constans II
  Ranked   #25: Constantine IV
  Ranked   #26: Justinian II Slit-Nose
  Ranked   #27: Leontios
  Ranked   #28: Tiberios III
  Ranked   #29: Philippikos
  Ranked   #30: Anastasios II
  Ranked   #31: Theodosios III
  Ranked   #32: Leo III the Isaurian
  Ranked   #33: Constantine V Kopronymos
  Ranked   #34: Ar

In [54]:
import pandas as pd
from IPython.display import HTML

df = pd.read_csv('ERE_List_20260226.csv')

rows = []
for _, r in df.iterrows():
    rows.append(
        f"<div style='display:flex;align-items:center;gap:14px;margin:4px 0;"
        f"border-bottom:1px solid #eee;padding:4px 0'>"
        f"<div style='width:36px;font-weight:bold'>#{r['List_Number']}</div>"
        f"<img src='{r['Local_Image']}' style='height:80px;width:64px;object-fit:cover'>"
        f"<div>{r['Simple_Name_With_Epithets']}<br>"
        f"<span style='color:#888;font-size:0.85em'>{r['Reign_Start']}–{r['Reign_End']}</span></div>"
        f"</div>"
    )
HTML(''.join(rows))

In [55]:
import pandas as pd
df = pd.read_csv('ERE_List_20260226.csv')
pairs = [(22,25),(35,37),(40,42),(62,71),(72,74),(77,79)]
for a, b in pairs:
    na = df.loc[df['List_Number']==a, 'Simple_Name_With_Epithets'].values[0]
    nb = df.loc[df['List_Number']==b, 'Simple_Name_With_Epithets'].values[0]
    pa = df.loc[df['List_Number']==a, 'Local_Image'].values[0]
    pb = df.loc[df['List_Number']==b, 'Local_Image'].values[0]
    print(f"#{a} {na} ({pa})  <->  #{b} {nb} ({pb})")

#22 Constantine III (images/022_.jpg)  <->  #25 Constantine IV (images/025_.jpg)
#35 Leo IV the Khazar (images/035_.jpg)  <->  #37 Eirene (images/037_.jpg)
#40 Michael I Rhangabe (images/040_.jpg)  <->  #42 Michael II the Amorian (images/042_.jpg)
#62 Isaac I Komnenos (images/062_.jpg)  <->  #71 Andronikos I Komnenos (images/071_.jpg)
#72 Isaac II Angelos (images/072_.png)  <->  #74 Alexios IV Angelos (images/074_.png)
#77 John III Doukas Vatatzes (images/077_.png)  <->  #79 John IV Laskaris (images/079_.png)
